# DuckDB GeoParquet Benchmarks

This notebook compares direct DuckDB queries across GeoParquet layouts: the raw
Microsoft Planetary Computer layout versus stac-hash-sorted layouts produced by
`cosgp convert`. It builds on the same `cosgp.cli.benchmark` framework that
backs the `cosgp benchmark run`/`compare` CLI commands, so results exported here
can be compared with `cosgp benchmark compare` directly.

For a single dataset with a fixed query suite, prefer the CLI:

```sh
uv run cosgp benchmark run my-dataset "optimized/*.parquet"
uv run cosgp benchmark compare benchmark-results/run-a-* benchmark-results/run-b-*
```

This notebook exists for deeper cross-layout comparisons that the CLI's
single-dataset, single-run model doesn't cover: running a fixed query suite
across several dataset variants at once, inspecting Parquet metadata, and
digging into `EXPLAIN ANALYZE` plans.

## Setup

Requires the `benchmark` extra and the `notebooks` dependency group:

```sh
uv sync --extra benchmark --group notebooks
uv run --group notebooks jupyter lab notebooks/duckdb-geoparquet-benchmarks.ipynb
```

In [1]:
from __future__ import annotations

import subprocess
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from stac_hash import Hasher

from cosgp.cli.benchmark.queries import (
    DEFAULT_REPEATS,
    QUERIES,
    BenchmarkSuite,
)
from cosgp.cli.benchmark.runner import (
    BenchmarkResult,
    BenchmarkRunner,
    queries_for_suite,
    runnable_queries,
    skipped_queries,
    sql_literal,
)

## Data Prep

Sync the raw layout from Source Cooperative, then generate hash-sorted variants
locally with `cosgp convert` instead of ad hoc scripts. Two variants isolate the
effect of sorting from the effect of file count:

- `hashed`: default bucket sizing (lets `cosgp convert` pick a natural file count)
- `hashed_matched_file_count`: `--match-file-count`, so it has the same file count
  as `microsoft`, keeping that variable fixed

This can take a while for the full dataset; it only needs to run once.

In [2]:
SOURCE_DIR = Path('../data/benchmarks/source/mspc-sentinel-2-l2a')
GENERATED_DIR = Path('../data/benchmarks/generated')

HASH_YEAR = 2025  # matches the sample data's Sentinel-2 L2A acquisition year
HASH_START_DATETIME = datetime(HASH_YEAR, 1, 1, tzinfo=UTC)
HASH_END_DATETIME = datetime(HASH_YEAR + 1, 1, 1, tzinfo=UTC)

BENCHMARK_PARAMS = {
    'collection': 'sentinel-2-l2a',
    'id': 'S2B_MSIL2A_20250101T031029_R075_T52VCJ_20250101T050301',
    'start_datetime': datetime(2025, 6, 1, tzinfo=UTC),
    'end_datetime': datetime(2025, 7, 1, tzinfo=UTC),
    'minx': -109.0,
    'miny': 37.0,
    'maxx': -102.0,
    'maxy': 41.0,
    'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
    'max_cloud_cover': 20.0,
}

In [3]:
SOURCE_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        'aws', 's3', 'sync',
        's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a/',
        str(SOURCE_DIR),
        '--no-sign-request', '--region', 'us-west-2',
        '--endpoint-url', 'https://s3.us-west-2.amazonaws.com',
        '--only-show-errors',
    ],
    check=True,
)

CompletedProcess(args=['aws', 's3', 'sync', 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a/', '../data/benchmarks/source/mspc-sentinel-2-l2a', '--no-sign-request', '--region', 'us-west-2', '--endpoint-url', 'https://s3.us-west-2.amazonaws.com', '--only-show-errors'], returncode=0)

In [4]:
def convert(outdir: Path, *extra_args: str) -> None:
    if outdir.exists():
        print(f'{outdir} already exists, skipping convert')
        return
    subprocess.run(
        [
            'cosgp', 'convert', str(SOURCE_DIR), str(HASH_YEAR), str(outdir),
            '--no-progress', *extra_args,
        ],
        check=True,
    )


convert(GENERATED_DIR / 'mspc-sentinel-2-l2a-hashed')
convert(GENERATED_DIR / 'mspc-sentinel-2-l2a-hashed-matched', '--match-file-count')

../data/benchmarks/generated/mspc-sentinel-2-l2a-hashed already exists, skipping convert
../data/benchmarks/generated/mspc-sentinel-2-l2a-hashed-matched already exists, skipping convert


## Dataset Variants

Keep all variants semantically equivalent (same items) so row counts match
across layouts. `REMOTE_DATASETS` points at the same raw layout hosted on
Source Cooperative; there's no hash-sorted layout hosted there yet, so the
remote run below is a latency-inclusive reference point rather than a full
cross-layout comparison.

In [5]:
DATASETS = {
    'microsoft': f'{SOURCE_DIR}/*.parquet',
    'hashed': f'{GENERATED_DIR}/mspc-sentinel-2-l2a-hashed/*.parquet',
    'hashed_matched_file_count': f'{GENERATED_DIR}/mspc-sentinel-2-l2a-hashed-matched/*.parquet',
}

SOURCE_COOPERATIVE_PREFIX = 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet'
REMOTE_DATASETS = {
    'remote_microsoft': f'{SOURCE_COOPERATIVE_PREFIX}/mspc-sentinel-2-l2a/*.parquet',
}

DATASETS, REMOTE_DATASETS

({'microsoft': '../data/benchmarks/source/mspc-sentinel-2-l2a/*.parquet',
  'hashed': '../data/benchmarks/generated/mspc-sentinel-2-l2a-hashed/*.parquet',
  'hashed_matched_file_count': '../data/benchmarks/generated/mspc-sentinel-2-l2a-hashed-matched/*.parquet'},
 {'remote_microsoft': 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a/*.parquet'})

## Query Suite

`QUERIES` comes straight from `cosgp.cli.benchmark.queries`, so the notebook
and CLI use the same benchmark definitions. `BENCHMARK_SUITE` controls whether
the notebook runs isolated `core` queries, multi-predicate `composite` queries,
or both. It defaults to the isolated `core` suite.

- Core queries isolate one filter or ordering dimension.
- Composite queries simulate realistic STAC searches with multiple predicates.
- Hash queries run only for datasets containing a `hash:hash` column.

Every unavailable query is reported with the missing columns or parameters.

In [6]:
BENCHMARK_SUITE = BenchmarkSuite.core
SELECTED_QUERIES = queries_for_suite(QUERIES, BENCHMARK_SUITE)

## Helpers

`BenchmarkRunner` (from the CLI framework) owns the DuckDB connection, query
timing, and `run.json` export. This notebook owns the fixed benchmark scenario
in `BENCHMARK_PARAMS`; for hash-sorted datasets it adds a real hash range for
that query window using `stac_hash.Hasher`, the same hasher `cosgp convert`
uses.

In [7]:
hasher = Hasher(HASH_START_DATETIME, HASH_END_DATETIME)


def dataset_columns(runner: BenchmarkRunner, path: str) -> set[str]:
    cursor = runner.connection.execute(
        f'DESCRIBE SELECT * FROM read_parquet({sql_literal(path)}) LIMIT 0'
    )
    return {row[0] for row in cursor.fetchall()}


def resolve_hash_range(params: dict[str, object]) -> tuple[int, int]:
    start = params['start_datetime'].astimezone(UTC)
    end = params['end_datetime'].astimezone(UTC)
    corners = [
        (params['minx'], params['miny']),
        (params['maxx'], params['miny']),
        (params['maxx'], params['maxy']),
        (params['minx'], params['maxy']),
    ]
    hashes = [
        hasher.hash_clamped(t, x, y) for t in (start, end) for x, y in corners
    ]
    return min(hashes), max(hashes)


def resolve_all_params(
    runner: BenchmarkRunner, path: str
) -> tuple[dict[str, object], set[str]]:
    params = dict(BENCHMARK_PARAMS)
    columns = dataset_columns(runner, path)
    if 'hash:hash' in columns:
        min_hash, max_hash = resolve_hash_range(params)
        params['min_hash'] = min_hash
        params['max_hash'] = max_hash
    return params, columns

## Parquet Metadata

Run this before timing queries. It verifies file counts, row groups, and
confirms the hash column is actually present and sorted where expected.

In [8]:
metadata_sql = """
SELECT
    file_name,
    count(DISTINCT row_group_id) AS row_groups,
    max(row_group_num_rows) AS max_row_group_rows,
    sum(row_group_compressed_bytes) AS compressed_bytes
FROM parquet_metadata({parquet_glob})
GROUP BY file_name
ORDER BY file_name
"""

runner = BenchmarkRunner(repeats=DEFAULT_REPEATS, progress=False)

for name, glob in DATASETS.items():
    print('\n##', name)
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    display(runner.connection.execute(sql).df())


## microsoft


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/source/mspc-sentinel-2-l2a/...,161,2048,1.010473e+11
1,../data/benchmarks/source/mspc-sentinel-2-l2a/...,144,2048,9.619865e+10
2,../data/benchmarks/source/mspc-sentinel-2-l2a/...,211,2048,1.346860e+11
3,../data/benchmarks/source/mspc-sentinel-2-l2a/...,226,2048,1.577516e+11
4,../data/benchmarks/source/mspc-sentinel-2-l2a/...,234,2048,1.592138e+11
5,../data/benchmarks/source/mspc-sentinel-2-l2a/...,225,2048,1.498662e+11
6,../data/benchmarks/source/mspc-sentinel-2-l2a/...,232,2048,1.465726e+11
7,../data/benchmarks/source/mspc-sentinel-2-l2a/...,231,2048,1.465465e+11
8,../data/benchmarks/source/mspc-sentinel-2-l2a/...,225,2048,1.437244e+11
9,../data/benchmarks/source/mspc-sentinel-2-l2a/...,215,2048,1.363797e+11



## hashed


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.383420e+10
1,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.301763e+10
2,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.305062e+10
3,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.331351e+10
4,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.196996e+10
5,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.220804e+10
6,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.220000e+10
7,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.405379e+10
8,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.344156e+10
9,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,4.334945e+10



## hashed_matched_file_count


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.809874e+10
1,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.700879e+10
2,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.794256e+10
3,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.576137e+10
4,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.590636e+10
5,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.847514e+10
6,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.760556e+10
7,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.777412e+10
8,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.861957e+10
9,../data/benchmarks/generated/mspc-sentinel-2-l...,3,150000,5.650861e+10


## Run Benchmarks

In [9]:
@dataclass
class DatasetRun:
    dataset_name: str
    dataset_path: str
    suite: BenchmarkSuite
    params: dict[str, object]
    skipped: dict[str, str]
    results: list[BenchmarkResult]

In [10]:
def run_dataset(
    runner: BenchmarkRunner, dataset_name: str, dataset_path: str
) -> DatasetRun:
    params, columns = resolve_all_params(runner, dataset_path)
    queries = runnable_queries(SELECTED_QUERIES, columns, params)
    skipped = skipped_queries(SELECTED_QUERIES, columns, params)
    for query_name, reason in skipped.items():
        print(f'{dataset_name} / {query_name}: skipped ({reason})')
    results = []
    for query in queries:
        try:
            result = runner.run_query(query, dataset_path, params)
        except Exception as error:
            print(f'{dataset_name} / {query.name}: {error}')
            continue
        results.append(result)
        print(
            f'{dataset_name} / {result.query}: rows={result.rows} '
            f'best={result.best_seconds:0.4f}s median={result.median_seconds:0.4f}s'
        )
    return DatasetRun(
        dataset_name, dataset_path, BENCHMARK_SUITE, params, skipped, results
    )


def run_matrix(runner: BenchmarkRunner, datasets: dict[str, str]) -> list[DatasetRun]:
    return [
        run_dataset(runner, name, path) for name, path in datasets.items()
    ]


def results_table(dataset_runs: list[DatasetRun]) -> pd.DataFrame:
    rows = [
        {
            'query': result.query,
            'dataset': run.dataset_name,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
        }
        for run in dataset_runs
        for result in run.results
    ]
    if not rows:
        return pd.DataFrame(
            columns=['query', 'dataset', 'rows', 'best_seconds', 'median_seconds']
        )
    return pd.DataFrame(rows).sort_values(['query', 'median_seconds']).reset_index(drop=True)

In [11]:
runs = run_matrix(runner, DATASETS)
summary = results_table(runs)
display(summary)

microsoft / hash_range_count: skipped (missing columns: hash:hash; missing parameters: min_hash, max_hash)
microsoft / hash_order_page: skipped (missing columns: hash:hash)
microsoft / full_dataset_count: rows=5031349 best=1.1768s median=1.1838s
microsoft / datetime_range_count: rows=459093 best=0.1546s median=0.1586s
microsoft / bbox_count: rows=8842 best=0.2107s median=0.2120s
microsoft / collection_count: rows=5031349 best=0.1712s median=0.1739s
microsoft / datetime_order_page: rows=100 best=0.1544s median=0.1614s
microsoft / cloud_cover_count: rows=1561626 best=0.1730s median=0.1759s
microsoft / geometry_intersects_count: rows=8842 best=0.9620s median=0.9821s
microsoft / id_lookup: rows=1 best=0.1705s median=0.1823s
hashed / full_dataset_count: rows=5031349 best=0.0201s median=0.0207s
hashed / datetime_range_count: rows=459093 best=0.0049s median=0.0050s
hashed / bbox_count: rows=8842 best=0.0091s median=0.0095s
hashed / collection_count: rows=5031349 best=0.0050s median=0.0051s
ha

,query,dataset,rows,best_seconds,median_seconds
0,bbox_count,hashed,8842,0.009120,0.009522
1,bbox_count,hashed_matched_file_count,8842,0.009861,0.010488
2,bbox_count,microsoft,8842,0.210681,0.211989
3,cloud_cover_count,hashed_matched_file_count,1561626,0.010084,0.010379
4,cloud_cover_count,hashed,1561626,0.010538,0.010950
5,cloud_cover_count,microsoft,1561626,0.173026,0.175867
6,collection_count,hashed_matched_file_count,5031349,0.004438,0.004891
7,collection_count,hashed,5031349,0.004978,0.005131
8,collection_count,microsoft,5031349,0.171160,0.173943
9,datetime_order_page,hashed_matched_file_count,100,0.003886,0.004096


### Remote Object-Store Run

Includes object-store listing, network latency, and HTTP range reads, so
compare it separately from the local-first results above. Set `RUN_REMOTE =
True` to run it; it's off by default since it needs network access and takes
longer.

In [12]:
RUN_REMOTE = False

if RUN_REMOTE:
    remote_runs = run_matrix(runner, REMOTE_DATASETS)
    remote_summary = results_table(remote_runs)
    display(remote_summary)

## Analyze Results

The speedup table compares median runtimes by query. `hashed` isolates the
sort effect together with whatever file count `cosgp convert` naturally
picked; `hashed_matched_file_count` isolates the sort effect alone by keeping
the file count equal to `microsoft`.

In [13]:
pivot = summary.pivot(index='query', columns='dataset', values='median_seconds')
speedups = pivot.copy()
if {'microsoft', 'hashed_matched_file_count'}.issubset(speedups.columns):
    speedups['microsoft_vs_hashed_matched_speedup'] = (
        speedups['microsoft'] / speedups['hashed_matched_file_count']
    )
if {'hashed', 'hashed_matched_file_count'}.issubset(speedups.columns):
    speedups['hashed_vs_hashed_matched_speedup'] = (
        speedups['hashed'] / speedups['hashed_matched_file_count']
    )

display(speedups.reset_index())

dataset,query,hashed,hashed_matched_file_count,microsoft,microsoft_vs_hashed_matched_speedup,hashed_vs_hashed_matched_speedup
0,bbox_count,0.009522,0.010488,0.211989,20.212156,0.907885
1,cloud_cover_count,0.010950,0.010379,0.175867,16.944816,1.055068
2,collection_count,0.005131,0.004891,0.173943,35.565354,1.049106
3,datetime_order_page,0.005108,0.004096,0.161374,39.394030,1.246852
4,datetime_range_count,0.004977,0.004405,0.158552,35.989883,1.129669
5,full_dataset_count,0.020717,0.016170,1.183799,73.210560,1.281216
6,geometry_intersects_count,0.834577,0.879105,0.982148,1.117214,0.949349
7,hash_order_page,0.004546,0.004015,NaN,NaN,1.132219
8,hash_range_count,0.004657,0.004179,NaN,NaN,1.114349
9,id_lookup,0.025404,0.024349,0.182340,7.488599,1.043306


In [14]:
metadata_rows = []
for dataset, glob in DATASETS.items():
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    df = runner.connection.execute(sql).df()
    metadata_rows.append({
        'dataset': dataset,
        'files': len(df),
        'row_groups': int(df['row_groups'].sum()),
        'compressed_gb': float(df['compressed_bytes'].sum() / 1_000_000_000),
        'median_row_groups_per_file': float(df['row_groups'].median()),
    })

file_summary = pd.DataFrame(metadata_rows).sort_values('dataset').reset_index(drop=True)
display(file_summary)

,dataset,files,row_groups,compressed_gb,median_row_groups_per_file
0,hashed,16,48,688.929832,3.0
1,hashed_matched_file_count,12,36,688.077515,3.0
2,microsoft,12,2463,1598.982122,220.0


### Reading the Summary

- `microsoft` vs `hashed_matched_file_count` keeps file count fixed, but still
  includes changes from rewriting compression and row-group layout.
- `hashed` vs `hashed_matched_file_count` shows how much file count itself
  changes timing for the same hash-sorted data.
- `full_dataset_count` is mostly a metadata/scan control; a big difference here
  usually means compression/schema/row-group overhead, not sorting.
- `composite_stac_search_count` is the main STAC-style query to watch; a hashed win
  here is the strongest signal that sorting helps.
- Hash-filter and hash-order queries are only meaningful on hash-sorted datasets;
  they are reported as skipped for `microsoft`.

Use `median_seconds` for comparisons; `best_seconds` is useful for spotting
warm-cache potential but can be optimistic.

## Inspect a Plan

Use this when a timing difference looks interesting. The key line to look
for is `Total Files Read`; if that number drops, DuckDB is pruning files.
Compares the STAC-style search against its hash-range equivalent on the
same hash-sorted dataset.

In [15]:
dataset_name = 'hashed_matched_file_count'
run = next(run for run in runs if run.dataset_name == dataset_name)

for query in SELECTED_QUERIES:
    if query.name not in (
        'composite_stac_search_count',
        'composite_hash_stac_search_count',
    ):
        continue
    sql = query.sql.format(
        **{
            key: sql_literal(value)
            for key, value in {'parquet_glob': DATASETS[dataset_name], **run.params}.items()
        }
    )
    print('\n##', query.name)
    print(sql)
    for row in runner.connection.execute('EXPLAIN ANALYZE ' + sql).fetchall():
        print(row[1] if len(row) > 1 else row[0])

## Export Results

Writes one `run.json` per dataset in the same format `cosgp benchmark run`
produces, so any pair can be compared directly with `cosgp benchmark
compare` — including comparisons against a run produced entirely by the
CLI rather than this notebook.

In [16]:
out_dir = Path('../benchmark-results')
out_dir.mkdir(parents=True, exist_ok=True)

for run in runs:
    run_file = runner.write(
        run.dataset_name,
        run.dataset_path,
        run.suite,
        run.params,
        run.skipped,
        run.results,
        out_dir,
    )
    print(f'wrote {run_file}')

if RUN_REMOTE:
    for run in remote_runs:
        run_file = runner.write(
            run.dataset_name,
            run.dataset_path,
            run.suite,
            run.params,
            run.skipped,
            run.results,
            out_dir,
        )
        print(f'wrote {run_file}')

wrote ../benchmark-results/microsoft-20260923T180347Z/run.json
wrote ../benchmark-results/hashed-20260923T180347Z/run.json
wrote ../benchmark-results/hashed_matched_file_count-20260923T180347Z/run.json
